In [16]:
import math
import matplotlib.pyplot as plt
import numpy as np
stages = [2,4,8] 

def easeOutBack(t, b=2, c=16):
    s = c * ((math.sin(t * (2 - t) * math.pi / 2)) ** 1) + b
    return min(stages, key=lambda x: abs(x - s))

for t in range(0, 8):
    # plot the function result
    print(easeOutBack(t/16))

2
4
8
8
8
8
8
8


In [35]:
stages = [0]
model_max_stage = 32
model_num = 4
for i in range(1, model_max_stage):
    if i%(model_num) == 0:
        stages.append(i)
stages

[0, 4, 8, 12, 16, 20, 24, 28]

In [42]:
from kubernetes import client, config

# Initialize the Kubernetes API client
config.load_kube_config()

kube_client = client.AppsV1Api()
autoscaling_client = client.AutoscalingV1Api()
v1 = client.CoreV1Api()

deployments = kube_client.list_namespaced_deployment(namespace='cdgp')
for deployment in deployments.items:
    # opt-66b-submod-8-latency-64 
    deployment_name = deployment.metadata.name
    if deployment_name.startswith('opt-66b-submod-8-latency-64'):
        anotation = deployment.metadata.annotations
        deployment.metadata.annotations["pipeline"] = "tinker-x"
        kube_client.patch_namespaced_deployment(name=deployment_name, namespace='cdgp', body=deployment)


In [44]:
import pandas as pd

class Pipeline:
    def __init__(self, namespace, kube_client, v1_client):
        self.namespace = namespace
        self.kube_client = kube_client
        self.v1_client = v1_client
        self.pipelines = self.update_pipeline_stage_pods()
        self.model_configs = pd.read_csv("model_configs.csv",names=["model","stages","inference_time"])
        self.pipeline_tolerance = 3
        self.pipeline_change_record = pd.DataFrame(columns=["pipeline","last_change_time"])
        self.pipeline_change_interval = 5*60

    def update_pipeline_stage_pods(self):
        # get all pod in namespace, get model name (pod_name.split('-') 0,1 is model name)
        pods = v1.list_namespaced_pod(namespace=self.namespace)
        pipeline_record = pd.DataFrame(columns=["model","pipeline","stages","pod_name","pod_ip"])
        for pod in pods.items:
            # get container 0 env
            env = pod.spec.containers[0].env
            # get model name from pod, get pipeline, stages from env, and append to pipeline_record if not exist
            model = pod.metadata.name.split('-')[0]
            pipeline = env["pipeline"].value
            stages = env["stages"].value
            pod_name = pod.metadata.name
            pod_ip = pod.status.pod_ip
            pipeline_record = pipeline_record.append({"model":model,"pipeline":pipeline,"stages":stages,"pod_name":pod_name,"pod_ip":pod_ip},ignore_index=True)            
        return pipeline_record
    
pipeline = Pipeline(namespace='cdgp', kube_client=kube_client, v1_client=v1)

TypeError: list indices must be integers or slices, not str

In [45]:
pods = v1.list_namespaced_pod('cdgp')
pipeline_record = pd.DataFrame(columns=["model","pipeline","stages","pod_name","pod_ip"])
for pod in pods.items:
    # get container 0 env
    env = pod.spec.containers[0].env
    # get model name from pod, get pipeline, stages from env, and append to pipeline_record if not exist
    model = pod.metadata.name.split('-')[0]
    pipeline = env["pipeline"].value
    stages = env["stages"].value
    pod_name = pod.metadata.name
    pod_ip = pod.status.pod_ip
    pipeline_record = pipeline_record.append({"model":model,"pipeline":pipeline,"stages":stages,"pod_name":pod_name,"pod_ip":pod_ip},ignore_index=True)            

TypeError: list indices must be integers or slices, not str

In [46]:
pod

{'api_version': None,
 'kind': None,
 'metadata': {'annotations': {'cni.projectcalico.org/containerID': 'a6fd5c5a728823615083a92fe82679eee0e43b7c169c3914ef1a5b23d9dd08b9',
                              'cni.projectcalico.org/podIP': '10.233.72.157/32',
                              'cni.projectcalico.org/podIPs': '10.233.72.157/32',
                              'k8s.v1.cni.cncf.io/network-status': '[{\n'
                                                                   '    '
                                                                   '"name": '
                                                                   '"k8s-pod-network",\n'
                                                                   '    "ips": '
                                                                   '[\n'
                                                                   '        '
                                                                   '"10.233.72.157"\n'
                              

In [47]:
env = pod.spec.containers[0].env

In [50]:
env

[{'name': 'debug', 'value': 'false', 'value_from': None},
 {'name': 'exec_timeout', 'value': '10.0s', 'value_from': None},
 {'name': 'fprocess', 'value': 'python index.py', 'value_from': None},
 {'name': 'infer_device', 'value': 'cuda', 'value_from': None},
 {'name': 'read_timeout', 'value': '10.0s', 'value_from': None},
 {'name': 'write_timeout', 'value': '10.0s', 'value_from': None},
 {'name': 'NVIDIA_VISIBLE_DEVICES',
  'value': 'GPU-7a2b56ed-eec1-de10-b054-ab35aefd7c73',
  'value_from': None},
 {'name': 'model_nums', 'value': '8', 'value_from': None}]

In [48]:
pod.metadata.name.split('-')[0]

'opt'

In [58]:
pipeline = [i.value for i in env if i.name == "NVIDIA_VISIBLE_DEVICES"]
pipeline

['GPU-7a2b56ed-eec1-de10-b054-ab35aefd7c73']

In [1]:
64//32


2